In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from rich.progress import Progress # Changed from 'track'
import numpy as np
import re
import torch.nn.functional as F
import os
import pickle

In [ ]:
text = open('shakespeare.txt').read().lower()

In [ ]:
text = re.sub(r'[^a-zA-Z\s]','',text)

words = text.split()

In [ ]:
len(words)

vocab = sorted(set(words))

wti = {word:i for i,word in enumerate(vocab)}
itw = {i:word for i,word in enumerate(vocab)}

In [ ]:
seq = 10

X = []
y = []

for i in range(len(words)-seq):
  X.append([wti[w] for w in words[i:i+seq]])
  y.append([wti[words[i+seq]]])

X = torch.tensor(X)
y = torch.tensor(y)

In [ ]:
class NextLSTM(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, 64)
    self.lstm = nn.LSTM(64, 128, batch_first=True)
    self.linear = nn.Linear(128, vocab_size)

  def forward(self,x):
    x = self.embedding(x)
    out, _ = self.lstm(x)
    out = out[:,-1,:]
    out = self.linear(out)

    return out

In [ ]:
model = NextLSTM(len(vocab))

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

### Moving Model and Data to GPU

In [ ]:
if os.path.exists("shakesphere.pkl"):
    with open("shakesphere.pkl", "rb") as file:
        model = pickle.load(file)
    print("Loaded pre-trained model from shakesphere.pkl")
else:
    print("No pre-trained model found. Training from scratch...")
    model.to(device)

    with Progress() as progress:
        epoch_task = progress.add_task("[green]Training Epochs[/green]", total=35)
        for epoch in range(35):
            batch_task = progress.add_task("[cyan]Batch Training[/cyan]", total=len(dataloader))
            for batch_X, batch_y in dataloader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)

                output = model.forward(batch_X)

                loss = loss_fn(output, batch_y.squeeze(-1))
                optimizer.zero_grad()

                loss.backward()
                optimizer.step()
                progress.update(batch_task, advance=1)
            progress.remove_task(batch_task)
            progress.update(epoch_task, advance=1)

In [ ]:
model.to(device)

with open("shakesphere.pkl", "wb") as file:
    pickle.dump(model, file)

In [ ]:
def generate_response(model,text,max_tokens=20, temperature=0.8, top_k=0, top_p=0.75):

  current_input_words = text.split()
  final_data_words = list(current_input_words)

  print(text)

  model.eval()
  with torch.no_grad():
    for i in range(max_tokens):
      model_input_words = current_input_words[-seq:]
      data = [wti[chunk.lower().strip()] for chunk in model_input_words]
      data = torch.tensor([data])
      data = data.to(device)

      output = model(data)

      probabilities = F.softmax(output / temperature, dim=-1)
      # print(probabilities)
      if top_p < 1.0:
          sorted_probs, sorted_indices = torch.sort(probabilities, descending=True)
          cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
          num_to_keep = (cumulative_probs < top_p).sum(dim=-1) + 1
          # print(num_to_keep)
          mask = torch.arange(probabilities.shape[-1], device=probabilities.device).unsqueeze(0) < num_to_keep.unsqueeze(1)
          # print(probabilities.shape[-1])
          filtered_sorted_probs = sorted_probs * mask
          # print(filtered_sorted_probs)

          probabilities = torch.zeros_like(probabilities).scatter_(-1, sorted_indices, filtered_sorted_probs)
          probabilities = probabilities / probabilities.sum(dim=-1, keepdim=True)
      elif top_k > 0:
          values, indices = torch.topk(probabilities, k=top_k, dim=-1)
          probabilities = torch.zeros_like(probabilities).scatter_(-1, indices, values)
          probabilities = probabilities / probabilities.sum(dim=-1, keepdim=True)
      # print(probabilities)



      word_idx = torch.multinomial(probabilities, 1).item()
      predicted_word = itw[word_idx]

      final_data_words.append(predicted_word)
      current_input_words.append(predicted_word)

  return ' '.join(final_data_words)

text = input('enter text: ').strip().lower()

generate_response(model,text,max_tokens=5, temperature=1, top_k=0, top_p=0.75)